In [1]:
from openai import OpenAI

# Create ollma endpoint
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
OLLAMA_KEY = 'ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_KEY)

In [2]:
system_msg = '''
You are a helpful airline ticket booking agent. Give short answers no more than 1 line. Always be very accurate. If you don't know the answer, say so.
'''

In [3]:
# Create tool

def ticket_price(destination_city):
    db = {'london': '65k INR', 'bali': '35k INR', 'tromso': '90k INR'}

    return db.get(destination_city.lower(), 'Unknown city')

# list of dictionaries. Each dictionary is a tool.
tools = [{
    'type': 'function',
    'function': {
        'name': 'ticket_price',
        'description': 'Get ticket price for given destination city',
        'parameters': {
            'type': 'object', # indicates to model that for function calling it should provide parameters in key-values pairs i.e. python dictionary
            'properties': { # Specifies each parameter of function
                'destination_city': {
                    'type': 'string',
                    'description': 'city cutomer wants to travel to',
                },
            },
            'required': ['destination_city'],
        },
    },
}]

In [6]:
import json

MODEL = 'deepseek-v3.1:671b-cloud' # One of the largest model in ollama. Runs on cloud.

def handle_tool_call(tool_calls):
    reply = []

    for tool in tool_calls: # Model may request multiple tool calls
        if tool.function.name == 'ticket_price':
            reply.append({
                'role': 'tool',
                'tool_call_id': tool.id,
                'content': ticket_price(json.loads(tool.function.arguments).get('destination_city')),
            })

    return reply
         

def chat(msg, history):
    # Gradio provide history of user and assistant conversation. Filter out role and content fields.
    history = [{'role': h['role'], 'content': h['content']} for h in history]

    # Prepare message for model. system msg + history of user/assitant conversation + user new message
    msg = [{'role' :'system', 'content': system_msg}] + history + [{'role': 'user', 'content': msg}]

    response = ollama.chat.completions.create(model=MODEL, messages=msg, tools=tools) # infer model

    # response.choices[0].finish_reason tells why model stopped generating text
    # e.g. max_tokens reached, tool call, natural stop etc
    while response.choices[0].finish_reason == 'tool_calls':
        '''
        Fields of response.choices[0].message:
            role: "assistant",
            content: "Hello! How can I help you today?",
            tool_calls:
                id: "call_abc123",
                type: "function",
                function: {
                    name: "ticket_price",
                    arguments: "{\"destination_city\": \"London\"}" (json)
        '''
        reply = handle_tool_call(response.choices[0].message.tool_calls)

        # Append model's response for tool call.
        # response.choices[0].message can be directly appended to chat.completions.create message. API handles it.
        msg.append(response.choices[0].message)

        msg.extend(reply) # Append our reply for all tool calls.

        print(msg)

        response = ollama.chat.completions.create(model=MODEL, messages=msg, tools=tools) # infer model again
    
    return response.choices[0].message.content

In [ ]:
import gradio as gr

gr.ChatInterface(fn=chat, type='messages').launch() # type='messages' records open AI complaint chat history

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


[{'role': 'system', 'content': "\nYou are a helpful airline ticket booking agent. Give short answers no more than 1 line. Always be very accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'Book me ticket to bali and tromso'}, ChatCompletionMessage(content='I can check ticket prices for those destinations for you.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_jgtf6vlx', function=Function(arguments='{"destination_city":"bali"}', name='ticket_price'), type='function', index=0)]), {'role': 'tool', 'tool_call_id': 'call_jgtf6vlx', 'content': '35k INR'}]
[{'role': 'system', 'content': "\nYou are a helpful airline ticket booking agent. Give short answers no more than 1 line. Always be very accurate. If you don't know the answer, say so.\n"}, {'role': 'user', 'content': 'Book me ticket to bali and tromso'}, ChatCompletionMessage(content='I can check ticket prices for t